In [9]:
import os
import re
import json
import time
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import clear_output

CATEGORY_NAMES = {
    "3.4": "✍️ 3.4 转写任务",
    "5.4": "⚖️ 5.4 法律行政",
    "5.7": "🎭 5.7 文化艺术",
    "6.1": "🛡️ 6.1 拒绝与边界",
    "6.2": "✅ 6.2 诚实与准确",
    "6.3": "🤝 6.3 价值观与尊重",
    "6.4": "💬 6.4 对话质量",
}


class RecentRejectionMonitor:
    def __init__(self, data_dir="output/data", active_timeout_seconds=300):
        self.data_dir = data_dir
        self.active_timeout_seconds = active_timeout_seconds
        self.state = {}
        self.reset()

    def _resolve_data_path(self):
        data_path = Path(self.data_dir)
        if not data_path.exists():
            data_path = Path("../") / self.data_dir
        if not data_path.exists():
            raise FileNotFoundError(f"找不到数据目录: {self.data_dir}")
        return data_path

    @staticmethod
    def _count_jsonl_lines(path):
        if not path.exists():
            return 0
        n = 0
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if line.strip():
                    n += 1
        return n

    @staticmethod
    def _sort_key(task_code):
        parts = []
        for x in task_code.split("."):
            try:
                parts.append(int(x))
            except ValueError:
                parts.append(9999)
        return parts

    @staticmethod
    def _display_width(text):
        text = re.sub(r"\033\[[0-9;]*m", "", text)
        return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in text)

    @staticmethod
    def _latest_mtime(*paths):
        mtimes = []
        for p in paths:
            if p.exists():
                mtimes.append(os.path.getmtime(p))
        return max(mtimes) if mtimes else 0

    @staticmethod
    def _rate_color(rate):
        if rate >= 50:
            return "\033[1;31m"
        if rate >= 25:
            return "\033[1;33m"
        return "\033[1;32m"

    @staticmethod
    def _add_reason(counter, reason):
        if isinstance(reason, dict):
            instr = reason.get("instruction")
            out = reason.get("output")

            if instr:
                counter[f"instruction:{instr}"] += 1
            if out:
                counter[f"output:{out}"] += 1
            if not instr and not out:
                counter[json.dumps(reason, ensure_ascii=False, sort_keys=True)] += 1
        else:
            counter[str(reason)] += 1

    def _read_new_rejects(self, path, old_line_count):
        counter = Counter()
        reason_counter = Counter()

        if not path.exists():
            return counter, reason_counter

        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for idx, line in enumerate(f):
                if idx < old_line_count:
                    continue

                line = line.strip()
                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    counter["bad_reject_json"] += 1
                    reason_counter["bad_reject_json"] += 1
                    continue

                stage = obj.get("stage", "quality")
                counter[stage] += 1
                self._add_reason(reason_counter, obj.get("reason"))

        return counter, reason_counter

    def _snapshot(self):
        data_path = self._resolve_data_path()
        snap = {}

        for task_dir in data_path.iterdir():
            if not task_dir.is_dir():
                continue

            task_code = task_dir.name
            if not re.match(r"^\d+(\.\d+)+$", task_code):
                continue

            data_file = task_dir / "data.jsonl"
            rejects_file = task_dir / "rejects.jsonl"

            snap[task_code] = {
                "data_lines": self._count_jsonl_lines(data_file),
                "reject_lines": self._count_jsonl_lines(rejects_file),
                "time": time.time(),
            }

        return snap

    def reset(self):
        """
        重置基线。
        下一次 check() 开始，只统计 reset 之后新增的成功和拒收。
        """
        self.state = self._snapshot()
        print("✅ 已重置 Notebook 内存基线。下一次 monitor.check() 将只统计从现在开始的新增拒收率。")

    def _fmt_task_line(self, task_code, d):
        active_mark = "↑" if d["is_active"] else " "
        color = self._rate_color(d["rate"])

        raw = (
            f"{task_code:<6}{active_mark} | "
            f"拒:{d['rate']:>5.1f}% "
            f"(+成:{d['gen']} +质:{d['quality']} +解:{d['parse']} +重:{d['duplicate']} +坏:{d['bad_json']})"
        )

        if d["is_active"]:
            colored = (
                f"\033[1;36m{task_code:<6}{active_mark}\033[0m | "
                f"拒:{color}{d['rate']:>5.1f}%\033[0m "
                f"(+成:{d['gen']} +质:{d['quality']} +解:{d['parse']} +重:{d['duplicate']} +坏:{d['bad_json']})"
            )
        else:
            colored = f"\033[90m{raw}\033[0m"

        return raw, colored

    def check(self, show_top_reasons=True, show_inactive_with_no_delta=False):
        """
        只统计最近新增拒收率：
            新增拒收 / (新增成功 + 新增拒收)

        新增拒收包括：
            quality + parse + duplicate + bad_reject_json
        """
        data_path = self._resolve_data_path()
        current_time = time.time()

        results = {}
        reason_results = {}
        new_state = {}

        for task_dir in data_path.iterdir():
            if not task_dir.is_dir():
                continue

            task_code = task_dir.name
            if not re.match(r"^\d+(\.\d+)+$", task_code):
                continue

            data_file = task_dir / "data.jsonl"
            rejects_file = task_dir / "rejects.jsonl"
            log_file = task_dir / "generation.log"

            current_data_lines = self._count_jsonl_lines(data_file)
            current_reject_lines = self._count_jsonl_lines(rejects_file)

            old = self.state.get(task_code, {
                "data_lines": current_data_lines,
                "reject_lines": current_reject_lines,
            })

            old_data_lines = int(old.get("data_lines", current_data_lines))
            old_reject_lines = int(old.get("reject_lines", current_reject_lines))

            delta_gen = max(0, current_data_lines - old_data_lines)
            delta_reject_lines = max(0, current_reject_lines - old_reject_lines)

            reject_counter, reason_counter = self._read_new_rejects(
                rejects_file,
                old_reject_lines,
            )

            quality = reject_counter.get("quality", 0)
            parse = reject_counter.get("parse", 0)
            duplicate = reject_counter.get("duplicate", 0)
            bad_json = reject_counter.get("bad_reject_json", 0)

            reject_total = quality + parse + duplicate + bad_json

            # 兜底：如果 rejects.jsonl 行数增加，但 stage 没解析出来，也不要漏掉。
            if reject_total == 0 and delta_reject_lines > 0:
                reject_total = delta_reject_lines

            attempts = delta_gen + reject_total
            rate = reject_total / attempts * 100 if attempts > 0 else 0.0

            mtime = self._latest_mtime(data_file, rejects_file, log_file)
            is_active = (current_time - mtime) <= self.active_timeout_seconds if mtime else False

            new_state[task_code] = {
                "data_lines": current_data_lines,
                "reject_lines": current_reject_lines,
                "time": current_time,
            }

            if attempts == 0 and not (show_inactive_with_no_delta and is_active):
                continue

            results[task_code] = {
                "gen": delta_gen,
                "quality": quality,
                "parse": parse,
                "duplicate": duplicate,
                "bad_json": bad_json,
                "reject_total": reject_total,
                "attempts": attempts,
                "rate": rate,
                "is_active": is_active,
            }
            reason_results[task_code] = reason_counter

        self.state = new_state

        clear_output(wait=True)

        if not results:
            print("📭 从上次 check() 到现在，没有新增成功或新增拒收。")
            print("需要重新设基线时运行：monitor.reset()")
            return

        grouped_tasks = defaultdict(list)
        for task_code in results:
            prefix = ".".join(task_code.split(".")[:2])
            grouped_tasks[prefix].append(task_code)

        print("=" * 120)
        print("🚀 SFT 任务拒收率体检报告：Notebook 最近新增版")
        print("📌 只统计上次 monitor.check() 到现在的新增数据，不显示历史累计。")
        print("📌 拒 = 质检拒 + 解析拒 + 重复拒 + 坏拒收日志。")
        print("=" * 120)

        total_gen = 0
        total_reject = 0
        active_gen = 0
        active_reject = 0

        for prefix in sorted(grouped_tasks.keys(), key=self._sort_key):
            tasks = sorted(grouped_tasks[prefix], key=self._sort_key)
            name = CATEGORY_NAMES.get(prefix, f"📁 {prefix} 其他任务")

            print(f"\n{name}")
            print("-" * 120)

            for i in range(0, len(tasks), 2):
                t1 = tasks[i]
                d1 = results[t1]

                total_gen += d1["gen"]
                total_reject += d1["reject_total"]

                if d1["is_active"]:
                    active_gen += d1["gen"]
                    active_reject += d1["reject_total"]

                col1_raw, col1_color = self._fmt_task_line(t1, d1)
                target_width = 58
                padding = max(0, target_width - self._display_width(col1_raw))

                if i + 1 < len(tasks):
                    t2 = tasks[i + 1]
                    d2 = results[t2]

                    total_gen += d2["gen"]
                    total_reject += d2["reject_total"]

                    if d2["is_active"]:
                        active_gen += d2["gen"]
                        active_reject += d2["reject_total"]

                    col2_raw, col2_color = self._fmt_task_line(t2, d2)
                    print(col1_color + (" " * padding) + " ||  " + col2_color)
                else:
                    print(col1_color)

            if show_top_reasons:
                reject_tasks = [t for t in tasks if results[t]["reject_total"] > 0]

                if reject_tasks:
                    worst_task = max(reject_tasks, key=lambda t: results[t]["rate"])
                    reasons = reason_results.get(worst_task, Counter())

                    if reasons:
                        print(f"  🔎 {worst_task} 最近主要拒收原因：")
                        for reason, n in reasons.most_common(6):
                            print(f"     - {n} × {reason}")

        print("\n" + "=" * 120)

        total_attempts = total_gen + total_reject
        total_rate = total_reject / total_attempts * 100 if total_attempts > 0 else 0.0

        print(
            f"🔥 【最近新增盘】 +成功:{total_gen} | +拒收:{total_reject} | "
            f"新增拒收率:{self._rate_color(total_rate)}{total_rate:.1f}%\033[0m"
        )

        active_attempts = active_gen + active_reject
        if active_attempts > 0:
            active_rate = active_reject / active_attempts * 100
            print(
                f"⚡ 【活跃新增盘】 +成功:{active_gen} | +拒收:{active_reject} | "
                f"活跃新增拒收率:{self._rate_color(active_rate)}{active_rate:.1f}%\033[0m"
            )

        print("=" * 120)

In [2]:
monitor = RecentRejectionMonitor(data_dir="output/data", active_timeout_seconds=300)

✅ 已重置 Notebook 内存基线。下一次 monitor.check() 将只统计从现在开始的新增拒收率。


In [11]:
monitor.check()

🚀 SFT 任务拒收率体检报告：Notebook 最近新增版
📌 只统计上次 monitor.check() 到现在的新增数据，不显示历史累计。
📌 拒 = 质检拒 + 解析拒 + 重复拒 + 坏拒收日志。

✅ 6.2 诚实与准确
------------------------------------------------------------------------------------------------------------------------
6.2.1 ↑ | 拒: 49.2% (+成:221 +质:3 +解:209 +重:2 +坏:0)    ||  6.2.2 ↑ | 拒: 64.9% (+成:135 +质:6 +解:179 +重:65 +坏:0)
6.2.3 ↑ | 拒: 70.4% (+成:100 +质:4 +解:234 +重:0 +坏:0)    ||  6.2.5 ↑ | 拒: 75.2% (+成:81 +质:0 +解:245 +重:1 +坏:0)
6.2.7 ↑ | 拒: 56.0% (+成:117 +质:2 +解:145 +重:2 +坏:0)
  🔎 6.2.5 最近主要拒收原因：
     - 245 × parse_json_failed
     - 1 × duplicate_assistant_content

💬 6.4 对话质量
------------------------------------------------------------------------------------------------------------------------
6.4.4 ↑ | 拒: 69.6% (+成:85 +质:3 +解:191 +重:1 +坏:0)     ||  6.4.8 ↑ | 拒: 67.0% (+成:145 +质:2 +解:293 +重:0 +坏:0)
  🔎 6.4.4 最近主要拒收原因：
     - 191 × parse_json_failed
     - 2 × instruction:too_short:3<8
     - 2 × output:too_short:3<16
     - 1 × instruction:chinese_contamination
  

In [10]:
import json
from pathlib import Path

def show_parse_fails(task_code, n=5):
    path = Path(f"output/data/{task_code}/rejects.jsonl")
    if not path.exists():
        print("no rejects.jsonl")
        return

    shown = 0
    for line in path.open(encoding="utf-8"):
        obj = json.loads(line)
        if obj.get("stage") == "parse":
            print("=" * 80)
            print("sample:", obj.get("sample_id"))
            print(obj.get("raw_output", "")[:2000])
            shown += 1
            if shown >= n:
                break

show_parse_fails("6.2.1", 5)
show_parse_fails("6.4.4", 5)

sample: bo_sft_004212
{
  "instruction": "ནག་ཆུ་རུ་མར་ལས་སྣོན་བཟོ་གྲྭ་ཞིག་བཙུགས་ན། མ་འོངས་ལོ་ལྔའི་ནང་གི་ཁེ་བཟང་གསལ་པོ་ག་ཚ
sample: bo_sft_004460
{
  "instruction": "ཉེ་རབས་ཀྱི་བླ་མ་དམ་པ་དེས་སྐུ་ཚེ་གང་པོར་ཐུགས་དམ་ཉམས་བཞེས་གནང་བའི་ཁྲོད་ཀྱི་གསང་བའི་
sample: bo_sft_008430
ད་གནང་བའི་གྲངས་ཐོ་འདྲ་མིན་ཡོད་སྲིད་མོད། བྱང་ཐང་གི་ས་བབ་མཐོ་དམའ་དང་གནམ་གཤིས་ཀྱི་འགྱུར་ལྡོག་
sample: bo_sft_007817
```json
{
  "instruction": "གནའ་བོའི་བོད་ཀྱི་མྱུལ་དཔྱད་པ་ཞིག་གིས་བྱང་ཐང་གི་མི་མེད་ལུང་སྟོང་བརྒལ་སྐབས། ཉིན་རེ་
sample: bo_sft_004516
ི་དཔེ་རྙིང་ཡིག་གཟུགས་ངོས་འཛིན་གྱི་རྙོག་འཛིང་རང་བཞིན་ལ་གཞིགས་ན། གསལ་ཚད་མཐོ་དམན་གྱི་བར་
sample: bo_sft_003579
{
  "instruction": "དེང་རབས་བོད་ཀྱི་རི་མོ་མཁས་ཅན་གྱི་བརྩམས་ཆོས་ཀྱི་ཁྱད་ཆོས་སྐོར་ཤོད་དང་།",
  "output": "མཚམས་སྦ
sample: bo_sft_002916
```json
{
  "instruction": "ལྷ་ས་ནས་ནང་སར་ཅ་ལག་བསྐུར་ན་སྦྲག་སྲིད་གང་ཞིག་གོང་ཁེ་ཤོས་རེད།",
  "output": "སྦ
sample: bo_sft_004166
范围内（如：དྲི་བ/问题、སེལ་བ/解决）".
    My words: ལམ་ཐིག (route), ལུས་སྟོབས (physical strength), འཚམ་པོ (suitable/fitting). Th